# SegmentTable Basics

This tutorial introduces `SegmentTable`, a container for segment-keyed analysis in **GWexpy**. Unlike a simple `EventTable`, `SegmentTable` is designed to hold not just metadata about time intervals, but also their corresponding payload (like TimeSeries or ASDs) with lazy-loading support.

In [ ]:
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable

# 1. Create simple segments
segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
st

## Delayed Loading with SegmentCell

You can add payload columns that are only loaded when needed. This is useful for handling large datasets.

In [ ]:
def my_loader():
    # Simulate loading data
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# Add a payload column with a loader (sequence of callables)
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

## Row-wise Processing

`SegmentTable` provides an `apply()` method to process each row and collect results into new columns.

In [ ]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

## Fetch and Conversion

You can explicitly load lazy cells with `fetch()` or `materialize()`. Converting to pandas gives you a standard DataFrame for meta columns.

In [ ]:
st2.fetch()
df = st2.to_pandas()
df.head()